# House Prices - ML (Kaggle competition)

## 1. Input the Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

data_train = pd.read_csv("/Users/dyk/Desktop/desktop/application/Kaggle/price_prediction/home-data-for-ml-course/train.csv")
data_test = pd.read_csv("/Users/dyk/Desktop/desktop/application/Kaggle/price_prediction/home-data-for-ml-course/test.csv")

print(f"size of the train set: {data_train.shape}\n"f"size of the test set: {data_test.shape}")

size of the train set: (1460, 81)
size of the test set: (1459, 80)


## 2. Feature Engineering

In [26]:
# (1)The total area of ​​the home’s outdoor leisure space. A larger area generally implies more functional and usable space, which may be positively correlated with the selling price.
data_train["TotalPorchSF"] =  (
    data_train["WoodDeckSF"] 
    + data_train["3SsnPorch"] #Three-Season Porch area in square feet
    + data_train["OpenPorchSF"] 
    + data_train["EnclosedPorch"]
    + data_train["ScreenPorch"]
)

# (2)The age of a property at the time of sale. Generally, an older property may command a lower price; however, its condition—in terms of renovations and maintenance—can also influence this relationship.
data_train["Houseage"] = data_train["YrSold"] - data_train["YearBuilt"]

# (3)The finished, usable area within a basement. This provides a better reflection of the actual space that can be lived in or utilized, compared to the "total basement area."
data_train["FinishedBsmtSF"] = data_train["BsmtFinSF1"] + data_train["BsmtFinSF2"]

# (4)An approximate indicator of the total area of ​​a property, which helps the model capture the overall scale and spatial capacity of the house.
data_train["TotalSF"] = (
    data_train["1stFlrSF"] 
    + data_train["TotalBsmtSF"] 
    + data_train["2ndFlrSF"] 
)


y = np.log1p(data_train["SalePrice"])
X_original = data_train.drop("SalePrice",axis = 1)

## 3. Data Preprocessing

**(2) permutation_importance**

In order to measure whether each variable is helpful for model prediction, permutation_importance is used. This method shuffles the values ​​of each feature in the validation set and then observes the changes of MAE (Mean Absolute Error). If MAE remains virtually unchanged, it suggests that the model's prediction do not rely on the specific feature which can be considered deleting.

Given that the higher corelated variables can potentially substitute for one another, shuffling any single feature then might result low importance score. Therefore, I iteratively remove features that cause minimal changes to the MAE, retrain the model, and compare the MEA changes. If the MAE remains stable, I wll proceed with the removal of those features.


In [7]:
X_train_full, X_valid_full, y_train_full, y_valid_full = train_test_split(
    X_original, y, train_size = 0.8, test_size = 0.2, random_state = 0
    )

#select categorical columns ***是否要删 with relatively low cardinality(convenient but arbitrary)
categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype == "object" ]

#select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ["int64", "float64"]]

my_cols = categorical_cols + numerical_cols

from sklearn.impute import SimpleImputer 
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder

#preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy="constant")
#preprocessing for categorical data
categorical_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))      #在处理validation data的时候忽略没有在training data里没有出现的data（新类别），防止报错，编码为0 0 0
    ])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

# define the training model
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=0)

my_pipeline = Pipeline(steps = [("preprocessor", preprocessor),
                                ("model", model)
                                ])

from sklearn.inspection import permutation_importance 
def feature_importance(X_train, X_valid, y_train, y_valid):

    my_pipeline.fit(X_train,y_train)

    result = permutation_importance(
        my_pipeline,
        X_valid,
        y_valid,
        scoring = "neg_mean_absolute_error",
        n_repeats = 10,
        random_state = 0
    )

    importance_df = pd.DataFrame({
        "Feature": X_valid.columns,
        "Importance_Mean": result.importances_mean,
        "Importance_Std": result.importances_std
    })

    return importance_df

importance_df = feature_importance(X_train_full, X_valid_full, y_train_full, y_valid_full).sort_values("Importance_Mean", ascending = True)

print(importance_df.round(4).to_string(index=False))

       Feature  Importance_Mean  Importance_Std
   FireplaceQu          -0.0003          0.0001
        MoSold          -0.0002          0.0002
     ExterCond          -0.0001          0.0001
         Fence          -0.0001          0.0001
  BsmtFinType2          -0.0001          0.0001
    Foundation          -0.0001          0.0002
            Id          -0.0001          0.0002
        Street          -0.0000          0.0000
  TotRmsAbvGrd          -0.0000          0.0001
      HalfBath          -0.0000          0.0002
     RoofStyle          -0.0000          0.0001
    BsmtFinSF2          -0.0000          0.0000
    GarageCond          -0.0000          0.0000
   LotFrontage          -0.0000          0.0003
       Heating          -0.0000          0.0000
    MasVnrType          -0.0000          0.0000
   MiscFeature           0.0000          0.0000
     Utilities           0.0000          0.0000
      PoolArea           0.0000          0.0000
     3SsnPorch           0.0000         

In [8]:
# Select features which are help for model prediction
feature_list = importance_df.loc[
    importance_df["Importance_Mean"] >= 0.0005,"Feature"
    ].unique()

In [9]:
# compute MAE with all features
from sklearn.model_selection import cross_val_score
scores = -1 * cross_val_score(my_pipeline, X_original, y,
                             cv = 5, 
                             scoring = "neg_mean_absolute_error")

print("Average MAE socre(across experiments):")
print(scores.mean())

Average MAE socre(across experiments):
0.09661304644592911


## 4. Model Training
In here, I used two ways to train model, which are Randomforest and XGboost. 

### (1) Randomforest

In [12]:
from sklearn.metrics import root_mean_squared_error
# compute MAE only with features which are help for model prediction
X = X_original[feature_list]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size = 0.8, test_size = 0.2, random_state = 0
    )

categorical_cols_new = [cname for cname in X.columns if X[cname].dtype == "object" ]
numerical_cols_new = [cname for cname in X.columns if X[cname].dtype in ["int64", "float64"]]

preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, numerical_cols_new),
        ("cat", categorical_transformer, categorical_cols_new)
    ])

leaf_list = [2 ** i for i in range(1,15)]
result = {}
for leaf_nodes in leaf_list:
    model = RandomForestRegressor(
        n_estimators=100,
        max_leaf_nodes = leaf_nodes,
        random_state=0)
    my_pipeline = Pipeline(steps = [("preprocessor", preprocessor),
                                ("model", model)
                                ])        
    my_pipeline.fit(X_train, y_train)
    preds_valid = my_pipeline.predict(X_valid)
    RMSE_valid = root_mean_squared_error(y_valid, preds_valid)
    result[leaf_nodes] = RMSE_valid
    print(f'leaf_nodes {leaf_nodes:6d},validate RMSE{RMSE_valid:8.4f}')

leaf_nodes      2,validate RMSE  0.2583
leaf_nodes      4,validate RMSE  0.1960
leaf_nodes      8,validate RMSE  0.1680
leaf_nodes     16,validate RMSE  0.1569
leaf_nodes     32,validate RMSE  0.1481
leaf_nodes     64,validate RMSE  0.1436
leaf_nodes    128,validate RMSE  0.1413
leaf_nodes    256,validate RMSE  0.1401
leaf_nodes    512,validate RMSE  0.1397
leaf_nodes   1024,validate RMSE  0.1397
leaf_nodes   2048,validate RMSE  0.1397
leaf_nodes   4096,validate RMSE  0.1397
leaf_nodes   8192,validate RMSE  0.1397
leaf_nodes  16384,validate RMSE  0.1397


We can tell the best leaf_nodes is 512

In [14]:
model = RandomForestRegressor(
        n_estimators=100,
        max_leaf_nodes = 512,
        random_state=0)

my_pipeline_new = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

scores_new = -1 * cross_val_score(my_pipeline_new, X, y,
                             cv = 5, 
                             scoring = "neg_mean_absolute_error")

print("Average MAE socre(across experiments):")
print(scores_new.mean())

Average MAE socre(across experiments):
0.09467338019907465


### (2) XGboost

In [16]:
from xgboost import XGBRegressor
model_XGB = XGBRegressor(
        n_estimators=491,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=3 or 5, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=0
        ) 

my_pipeline_XGB = Pipeline(steps = [("preprocessor", preprocessor),
                                ("model", model_XGB)
                                ])

scores_XGB = -1 * cross_val_score(my_pipeline_XGB, X, y,
                             cv = 5, 
                             scoring = "neg_mean_absolute_error")    

print("Average MAE socre(across experiments) using XGBoost:")
print(scores_XGB.mean())                                                      



Average MAE socre(across experiments) using XGBoost:
0.0816847048241249


##### In conclusion: From MAE score, XGboost model is better